# 02. PDF를 Markdown으로 변환한 RAG 개선 실습

## 실습 목적

이 실습에서는 PDF 문서를 바로 로딩하지 않고 Markdown 문서로 변환한 뒤 RAG를 구성한다.  
PDF 직접 로딩 방식과 비교하여 문서 구조, 표, 제목 정보가 검색 품질에 어떤 영향을 주는지 확인한다.

## Advanced RAG 개선 방향
```
Naive RAG:    질문 → 검색 → 생성
Advanced RAG: 
- PDF를 Markdown으로 변환하여 문서 구조를 보존한 뒤 로딩
- 질문 → [쿼리 재작성 / 다중쿼리] → 검색 → [재순위화 / 압축] → 생성
```

## 학습 목표

이번 실습에서는 PDF 문서를 Markdown으로 변환한 뒤 RAG를 구성하고, PDF 직접 로딩 방식과 Markdown 기반 로딩 방식의 차이를 확인한다.

실습 후 다음 내용을 설명할 수 있어야 한다.

1. PDF 문서를 Markdown으로 변환하는 이유
2. Markdown 변환 후에도 정제가 필요한 이유
3. Markdown 문서를 `TextLoader`로 로딩하는 방법
4. Markdown 기반 청크 분할 방식의 특징
5. PDF 기반 RAG와 Markdown 기반 RAG의 검색 결과 차이
6. Markdown 변환 방식이 표, 제목, 문단 구조 보존에 미치는 영향

## 실습 데이터
**「2026 AI 주요 트렌드 전망」NIA AI 트렌드 보고서 기반 한국어 RAG 챗봇 만들기** 

In [1]:
# 추가 설치 필요 패키지
# uv add python-dotenv pymupdf4llm chromadb langchain-community langchain-text-splitters langchain-openai langchain-chroma

from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

True

# pdf -> Markdown 변환
- Docling: 테이블 보존과 RAG 전처리 설명에 적합
- PyMuPDF4LLM: 코드가 가장 단순해 입문 실습에 좋음
- Marker: 고급 실습에서 LLM 보정, 복잡한 표 처리까지 확장하기 좋음
- uv add docling

## pdf를 markdown 변환

In [ ]:
from pathlib import Path
import pymupdf4llm

PDF_PATH = Path("data/AI@Data_Report_2026_전망_251223(최종).pdf")
MD_PATH = PDF_PATH.with_suffix(".md")

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {PDF_PATH}")

# --------------------------------------------------
# PDF를 Markdown으로 변환해서 저장
# --------------------------------------------------

md_text = pymupdf4llm.to_markdown(str(PDF_PATH))
MD_PATH.write_text(md_text, encoding="utf-8")

print(f"Markdown 파일 저장 완료: {MD_PATH}")
print(f"Markdown 미리보기:\n{md_text[:500]}")

Markdown 파일 저장 완료: data\AI@Data_Report_토픽_분석을_통한_AI_주요_트렌드_및_2026_전망_251223(최종).md
Markdown 미리보기:
AI@Data Report **토픽 분석을 통한 AI 주요 트렌드 및 2026 전망 2025-제3호** 

**목차** 

❶ **배경 및 목적 /p.06** 

❷ **자료 분석 및 방법 /p.09** 

- ❸ **분석 결과 /p.12** 

- ❹ **2026년 전망: AI 생태계 방향성 /p.17** 

❺ **분석 결과로 본 우리의 대응 및 노력 /p.20** 

## AI@Data Report 

**2025 제 3 호 (2025. 12월)** 

## 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망 

## **AI@Data 보고서 시리즈는...** 

**AI와 Data 관련된 국내외 기술 및 정책 동향과 이슈 분석을 통해 미래 사회의 트렌드를 주도할 수 있는 정책 아젠다를 제시하기 위해 한국지능정보사회진흥원(NIA)에서 기획, 발간하는 보고서입니다.** 

- ❶ 본 보고서는 과학기술정보통신부와 한국지능정보사회진흥원(NIA)의 

- ‘초거대 AI 데이터 발굴 


## md 파일 전처리하기

In [3]:
from pathlib import Path

# 전처리 스크립트 파일 존재 여부 확인
PREPROCESS_FILE = Path("markdown_pre_processing.py")
if not PREPROCESS_FILE.exists():
    raise FileNotFoundError(
        "markdown_pre_processing.py 파일을 찾을 수 없습니다. "
        "이 파일을 현재 노트북과 같은 폴더에 두세요."
    )
# markdown 전처리
from markdown_pre_processing import clean_markdown
RAW_MD_PATH = Path("data/AI@Data_Report_토픽_분석을_통한_AI_주요_트렌드_및_2026_전망_251223(최종).md")
CLEAN_MD_PATH = Path("data/AI@Data_Report_CLEANED.md")
clean_markdown(RAW_MD_PATH, CLEAN_MD_PATH)

정제된 Markdown 저장 완료: data\AI@Data_Report_CLEANED.md
AI@Data Report **
-제3호** 

**목차** 

❶ **배경 및 목적 /p.06** 

❷ **자료 분석 및 방법 /p.09** 

- ❸ **분석 결과 /p.12** 

- ❹ **2026년 전망: AI 생태계 방향성 /p.17** 

❺ **분석 결과로 본 우리의 대응 및 노력 /p.20** 

## AI@Data Report 

**2025 제 3 호 (2025. 12월)** 

## 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망 

## **AI@Data 보고서 시리즈는...** 

**AI와 Data 관련된 국내외 기술 및 정책 동향과 이슈 분석을 통해 미래 사회의 트렌드를 주도할 수 있는 정책 아젠다를 제시하기 위해 한국지능정보사회진흥원(NIA)에서 기획, 발간하는 보고서입니다.** 

- ❶ 본 보고서는 과학기술정보통신부와 한국지능정보사회진흥원(NIA)의 

- ‘초거대 AI 데이터 발굴 및 기획’ 사업의 산출물이므로, 본 보고서의 내용을 발표할 때는 반드시 과학기술정보통신부 사업의 연구결과임을 밝혀야합니다. 

- ‧ 

- ❷ 보고서 내용의 무단전재를 금하며, 가공 인용할 때는 반드시 출처를 

- 「한국지능정보사회진흥원(NIA)」이라고 밝혀주시기 바랍니다. 

- ❸ 본 보고서의 내용은 한국지능정보사회진흥원(NIA)의 

- 공식 견해와 다를 수 있습니다. 

**작성  한국지능정보사회진흥원** 김시연 수석(siyeon@nia.or.kr, 053-230-4249) 추지혜 책임(chuu@nia.or.kr, 053-230-4247) 전민석 주임(msj1224@nia.or.kr, 053-230-4205) 

**기획  한국지능정보사회진흥원 인공지능데이터본부** 신신애 본부장(sashin@nia.or.kr, 053-230-4200) 심호찬 팀장(shc@nia.or.kr, 053-230-4201) 

## **표·그림 목차** 

|**[

# ChromaDB로 Vector DB 구성

ChromaDB는 로컬 파일에 영구 저장(persist)이 가능한 오픈소스 벡터 DB입니다.

In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from pathlib import Path

CLEAN_MD_PATH = Path("data/AI@Data_Report_CLEANED.md")

if not CLEAN_MD_PATH.exists():
    raise FileNotFoundError(f"정제된 Markdown 파일을 찾을 수 없습니다: {CLEAN_MD_PATH}")

loader = TextLoader(str(CLEAN_MD_PATH), encoding="utf-8")
docs = loader.load()

print(f"로드된 문서 수: {len(docs)}")
print("\n[문서 metadata]")
print(docs[0].metadata)

print("\n[문서 미리보기]")
print(docs[0].page_content[:800])

로드된 문서 수: 1

[문서 metadata]
{'source': 'data\\AI@Data_Report_CLEANED.md'}

[문서 미리보기]
AI@Data Report **
-제3호** 

**목차** 

❶ **배경 및 목적 /p.06** 

❷ **자료 분석 및 방법 /p.09** 

- ❸ **분석 결과 /p.12** 

- ❹ **2026년 전망: AI 생태계 방향성 /p.17** 

❺ **분석 결과로 본 우리의 대응 및 노력 /p.20** 

## AI@Data Report 

**2025 제 3 호 (2025. 12월)** 

## 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망 

## **AI@Data 보고서 시리즈는...** 

**AI와 Data 관련된 국내외 기술 및 정책 동향과 이슈 분석을 통해 미래 사회의 트렌드를 주도할 수 있는 정책 아젠다를 제시하기 위해 한국지능정보사회진흥원(NIA)에서 기획, 발간하는 보고서입니다.** 

- ❶ 본 보고서는 과학기술정보통신부와 한국지능정보사회진흥원(NIA)의 

- ‘초거대 AI 데이터 발굴 및 기획’ 사업의 산출물이므로, 본 보고서의 내용을 발표할 때는 반드시 과학기술정보통신부 사업의 연구결과임을 밝혀야합니다. 

- ‧ 

- ❷ 보고서 내용의 무단전재를 금하며, 가공 인용할 때는 반드시 출처를 

- 「한국지능정보사회진흥원(NIA)」이라고 밝혀주시기 바랍니다. 

- ❸ 본 보고서의 내용은 한국지능정보사회진흥원(NIA)의 

- 공식 견해와 다를 수 있습니다. 

**작성  한국지능정보사회진흥원** 김시연 수석(siyeon@nia.or.kr, 053-230-4249) 추지혜 책임(chuu@nia.or.kr, 053-230-4247) 전민석 주임(msj1224@nia.or.


## 청크 분할

In [5]:
# Markdown 문서는 전체 파일이 하나의 Document로 로딩된다.
# 따라서 검색 결과를 추적하기 위해 chunk_id를 metadata에 추가한다.

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = splitter.split_documents(docs)

for i, chunk in enumerate(chunks, start=1):
    chunk.metadata["chunk_id"] = i
    chunk.metadata["doc_type"] = "markdown"
    chunk.metadata["source_name"] = CLEAN_MD_PATH.name

print(f"생성된 chunk 수: {len(chunks)}")

print("\n[첫 번째 chunk metadata]")
print(chunks[0].metadata)

print("\n[첫 번째 chunk 내용]")
print(chunks[0].page_content[:500])

생성된 chunk 수: 29

[첫 번째 chunk metadata]
{'source': 'data\\AI@Data_Report_CLEANED.md', 'chunk_id': 1, 'doc_type': 'markdown', 'source_name': 'AI@Data_Report_CLEANED.md'}

[첫 번째 chunk 내용]
AI@Data Report **
-제3호** 

**목차** 

❶ **배경 및 목적 /p.06** 

❷ **자료 분석 및 방법 /p.09** 

- ❸ **분석 결과 /p.12** 

- ❹ **2026년 전망: AI 생태계 방향성 /p.17** 

❺ **분석 결과로 본 우리의 대응 및 노력 /p.20** 

## AI@Data Report 

**2025 제 3 호 (2025. 12월)** 

## 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망 

## **AI@Data 보고서 시리즈는...** 

**AI와 Data 관련된 국내외 기술 및 정책 동향과 이슈 분석을 통해 미래 사회의 트렌드를 주도할 수 있는 정책 아젠다를 제시하기 위해 한국지능정보사회진흥원(NIA)에서 기획, 발간하는 보고서입니다.** 

- ❶ 본 보고서는 과학기술정보통신부와 한국지능정보사회진흥원(NIA)의 

- ‘초거대 AI 데이터 발굴 및 기획’ 사업의 산출물이므로, 본 보고서의 내용을 발표할 


## ChromaDB 생성

In [6]:
import shutil
from pathlib import Path

# 4. Embedding + Chroma 저장
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
    )

DB_PATH = Path("chroma_db/ai_trend_report_md")
COLLECTION_NAME = "ai_trend_report_2026_md"

# 실습에서는 매번 깨끗하게 다시 생성
if DB_PATH.exists():
    shutil.rmtree(DB_PATH)

# ChromaDB 생성
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=str(DB_PATH)
)

print(f"저장된 청크 수: {vector_store._collection.count()}")

저장된 청크 수: 29


## 기존 ChromaDB 불러오기

In [7]:
# 이미 저장된 DB를 불러올 때
loaded_store = Chroma(
    embedding_function=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=str(DB_PATH)
)

print(f"불러온 청크 수: {loaded_store._collection.count()}")

불러온 청크 수: 29


# Markdown 기반 RAG 검색 품질 확인

In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
retriever = loaded_store.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

prompt = ChatPromptTemplate.from_messages([
    ("system", "아래 문서를 참고해서 질문에 답하세요.\n\n[문서]\n{context}"),
    ("human", "{question}")
])

naive_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

In [9]:
### 표현 방식에 따른 검색 편차
# [질문]
# 1. 2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?
# 2. 내년 AI 모델은 어떤 방향으로 발전할 것으로 보이나요?
# 3. 합성데이터, 추론형 AI, 멀티모달 기술은 앞으로 어떤 역할을 하나요?
# 4. AI 정책 분야에서 안전성과 책임이 왜 중요한 이슈인가요?
# 5. 신뢰할 수 있는 AI를 만들기 위해 어떤 제도적 장치가 필요하다고 하나요?


In [10]:
def inspect_retrieval(query: str, k: int = 5, active_store=loaded_store):
    """
    질문에 대해 검색된 문서를 확인한다.

    ChromaDB의 similarity_search_with_score()에서 반환되는 score는 거리값에 가깝다.
    일반적으로 낮을수록 질문과 문서가 더 가깝다.
    """

    results = active_store.similarity_search_with_score(query, k=k)

    print("=" * 100)
    print(f"[질문] {query}")
    print(f"[검색 개수 k] {k}")
    print("ChromaDB distance score: 낮을수록 더 유사함")
    print("=" * 100)

    for i, (doc, score) in enumerate(results, start=1):
        source = doc.metadata.get("source", "?")
        chunk_id = doc.metadata.get("chunk_id", "?")
        doc_type = doc.metadata.get("doc_type", "?")
        source_name = doc.metadata.get("source_name", "?")

        print(f"\n[{i}] distance={score:.4f} | chunk_id={chunk_id} | doc_type={doc_type}")
        print(f"source={source}")
        print(f"source_name={source_name}")
        print(doc.page_content[:600].replace("\n", " "))

## 표현 방식에 따른 검색 편차

다음 질문들은 의미적으로 유사하지만 표현 방식이 다르다.  
RAG 검색은 질문 문장 자체를 임베딩하므로, 같은 의도라도 표현이 달라지면 검색 결과가 달라질 수 있다.


확인 질문:

1. 2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?
2. 내년 AI 모델은 어떤 방향으로 발전할 것으로 보이나요?
3. 합성데이터, 추론형 AI, 멀티모달 기술은 앞으로 어떤 역할을 하나요?
4. AI 정책 분야에서 안전성과 책임이 왜 중요한 이슈인가요?
5. 신뢰할 수 있는 AI를 만들기 위해 어떤 제도적 장치가 필요하다고 하나요?

In [11]:
# 문제 1: 표현 방식에 따른 검색 편차
q1 = "2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?"
q2 = "내년 AI 모델은 어떤 방향으로 발전할 것으로 보이나요?"

inspect_retrieval(q1, k=3)
print("-" * 80)

inspect_retrieval(q2, k=3)

[질문] 2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?
[검색 개수 k] 3
ChromaDB distance score: 낮을수록 더 유사함

[1] distance=0.7850 | chunk_id=21 | doc_type=markdown
source=data\AI@Data_Report_CLEANED.md
source_name=AI@Data_Report_CLEANED.md
## **4.1 2025년 AI 산업·기술·정책의 트렌드 현황**   - 앞 장에서는 2025년 주요 매체 데이터를 바탕으로 산업·기술·정책 전반에서 나타난 AI 담론의 구조적 흐름을 도출하였음   - - 분석 결과를 통해 세 분야는 서로 다른 영역에서 출발하더라도 활용 확대 → 기술적 고도화 요구 증가 → 제도적 대응 심화 로 이어지는 연속적 흐름을 형성하고 있음을 확인하였음   - 2026년에는 이 세 방향성이 더욱 맞물려 산업 구조의 재배치, 기술 체계의 정교화, 규제·책임 체계의 본격적 적용으로 이어지는 개략적 전망을 제시함   ## **4.2 2026년 AI 산업 전망**   2026년 AI 시장은 선도 기업을 중심으로 내재화가 빠르게 진행 되며 시장 규모가 지속 확대될 전망   - - AI 활용이 여러 산업군에서 실험 단계를 넘어 일부 핵심 공정과 서비스 운영에 본격 도입되기 시작하면서 AI가 기업 운영 인프라로 자리 잡아가는 흐름이 확산될 가능성이 높음   - - 특히 대기업·디지털 강도가 높은 산업을 중심으로 의사결정·운영·고객 대응 등 주요 영역에 AI가 단계적으로 통합되며 AI 관련 시장 수요가 꾸준한 성장세를 이어갈 것으로 평가됨   - 

[2] distance=0.8094 | chunk_id=6 | doc_type=markdown
source=data\AI@Data_Report_CLEANED.md
source_name=AI@Data_Report_CLEANED.md
- - 산업·정책 영역에서는 제도화 지연에 따른 불확실성과 대응 부담이 증가하고 있으며, 국가·기

## 관련성이 낮은 청크도 함께 반환되는 문제

기본 검색 방식은 `k=5`로 설정하면 관련성이 낮은 문서가 포함되더라도 최대 5개의 청크를 반환한다.  
이 경우 LLM 프롬프트에 불필요한 정보가 함께 들어가 답변 품질이 낮아질 수 있다.

확인 질문:

1. AI 에이전트가 기업 운영 방식에 어떤 변화를 가져올 것으로 전망되나요?
2. 보고서에서 말하는 온디바이스 AI 확산은 어떤 의미인가요?
3. 토픽모델링에서 단순 출현 빈도보다 중요하게 본 기준은 무엇인가요?
4. AI 정책에서 출력물 표시와 데이터 출처 공개는 왜 중요하게 다루어지나요?
5. 보고서의 전처리 과정에서 제거한 단어 유형은 무엇인가요?

In [12]:
### 관련 없는 청크도 반환
# [질문]
# 1. AI 에이전트가 기업 운영 방식에 어떤 변화를 가져올 것으로 전망되나요?
# 2. 보고서에서 말하는 온디바이스 AI 확산은 어떤 의미인가요?
# 3. 토픽모델링에서 단순 출현 빈도보다 중요하게 본 기준은 무엇인가요?
# 4. AI 정책에서 출력물 표시와 데이터 출처 공개는 왜 중요하게 다루어지나요?
# 5. 보고서의 전처리 과정에서 제거한 단어 유형은 무엇인가요?

### 크로마 db의 유사도
- 유사도 계산후 반환 되는 값이 거리값, 값이 작을 수록 유사함.
- 유클리드 거리 제곱 = Squared L2 Distance, 벡터 간 거리의 제곱. 기본값
- cosine 기반 거리를 사용하려면 명시적으로 지정해야함. 

In [13]:
# 문제 2: 관련 없는 청크도 반환
query =  "AI 에이전트가 기업 운영 방식에 어떤 변화를 가져올 것으로 전망되나요?"

print("유사도 거리 점수 (낮을수록 유사):")
inspect_retrieval(query, k=5)

유사도 거리 점수 (낮을수록 유사):
[질문] AI 에이전트가 기업 운영 방식에 어떤 변화를 가져올 것으로 전망되나요?
[검색 개수 k] 5
ChromaDB distance score: 낮을수록 더 유사함

[1] distance=1.0193 | chunk_id=22 | doc_type=markdown
source=data\AI@Data_Report_CLEANED.md
source_name=AI@Data_Report_CLEANED.md
- - 이와 함께 기존 산업의 가치 사슬도 전면 재편이라기보다는 핵심 공정·고부가 서비스 단에서 데이터·AI 중심 구조로 이동하는 압력이 강화되는 양상을 보일 것으로 전망됨   - 기업 내부에서 AI 에이전트를 활용한 문서 처리, 고객 지원, 운영 자동화 등이 증가하며 – –   - 사람 에이전트 시스템이 혼합된 업무 구조 가 일부 영역에서 확산될 가능성이 있음   - - 기업 간(B2B) 시스템 연동을 통해 제한된 범위에서 AI 간 협업 형태의 자동화 사례도 등장할 것으로 예상되지만, 그 영향은 아직 초기 단계에 머물며 중장기적 구조 변화의 단초를 형성하는 수준으로 평가됨   - ※ IoT 센서와 AI 기술을 통해 물류 및 공급망 전 과정을 자동화하고 최적화[4)]   ## **[ 그림 5 ] 주요 산업 분야별 AI 적용 사례**   ## **4.3 2026년 AI 기술 전망**   - 2026년 AI 기술은 지능 구조 고도화 와 데이터 한계 보완을 중심 으로 진화할 전망임   - - 합성데이터·추론형 AI·멀티모달 기술이 주요 경쟁 축으로 자리 잡으면서, 학습 효율 향상·복합 정보 처리·설명 가능성 강화 등 모델 내부 구조의 질적 개선 흐름이 이어질 것으

[2] distance=1.0333 | chunk_id=19 | doc_type=markdown
source=data\AI@Data_Report_CLEANED.md
source_name=AI@Data_Report_CLEANED.md
## **3.2.4 [AI 산업·기술·정

## 표·절차 정보 검색 확인

Markdown 변환은 PDF 직접 로딩보다 표와 제목 구조를 더 잘 보존할 수 있다.  
그러나 표가 길거나 청크 경계가 잘못 나뉘면 행·열 관계가 여전히 깨질 수 있다.

확인 질문:

1. 이 보고서는 AI 트렌드를 분석하기 위해 어떤 분석 절차를 거쳤나요?
2. 산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?
3. 토픽모델링 전에 제거한 전처리 용어 유형과 예시를 정리해줘.
4. AI 산업, AI 기술, AI 정책의 토픽명과 핵심 이슈를 비교해줘.
5. 추론형 데이터는 어떤 유형으로 구분되며, 각 유형의 예시는 무엇인가요?

In [14]:
## 표·절차 정보 검색 확인
# [질문]
# 1. 이 보고서는 AI 트렌드를 분석하기 위해 어떤 분석 절차를 거쳤나요?
# 2. 산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?
# 3. 토픽모델링 전에 제거한 전처리 용어 유형과 예시를 정리해줘.
# 4. AI 산업, AI 기술, AI 정책의 토픽명과 핵심 이슈를 비교해줘.
# 5. 추론형 데이터는 어떤 유형으로 구분되며, 각 유형의 예시는 무엇인가요?

In [15]:
query =  "이 보고서는 AI 트렌드를 분석하기 위해 어떤 분석 절차를 거쳤나요?"

print("유사도 거리 점수 (낮을수록 유사):")
inspect_retrieval(query, k=5)

유사도 거리 점수 (낮을수록 유사):
[질문] 이 보고서는 AI 트렌드를 분석하기 위해 어떤 분석 절차를 거쳤나요?
[검색 개수 k] 5
ChromaDB distance score: 낮을수록 더 유사함

[1] distance=0.7849 | chunk_id=9 | doc_type=markdown
source=data\AI@Data_Report_CLEANED.md
source_name=AI@Data_Report_CLEANED.md
- 기사·보고서 기반 분석은 실제 AI 트렌드의 전체 맥락과 세부 이슈를 완전히 포괄하기 어려우므로, 향후 연구 및 정책 제안에서는 보다 다양한 데이터 소스 확충과 다층적 분석, 심층 사례 검토가 필요함   - - 언론의 이슈 편향, 국가별 리스크 인식 차이, 기업 홍보성 보도 등으로 인해 담론 기반 분석은 구조적 편향(Structural Bias)이 발생할 여지가 있음   - - 동일 키워드라도 국가별·산업별 맥락이 매우 상이할 수 있어 일반화에는 주의가 필요함   - 분석 결과는 현재 시점의 데이터에 기반하므로, 향후 기술 발전·정책 변화·시장 흐름에 따라 주요 이슈가 재편될 수 있으며, 지속적 모니터링이 필수적임   - - AI 규제체계, 에이전틱 AI, 글로벌 인프라 경쟁 등은 변화 속도가 빨라, 정기적인 재분석 및 업데이트가 요구됨   ## CHAPTER2 **분석 절차**   ## **2.1 자료 수집 절차 및 분석 과정**   - 각 분야별 담론 구조 도출을 위해 본 연구는 데이터 기반 접근법을 적용하였으며, 그 분석 과정은 다음의 단계별 프로세스에 따라 수행되었음   ## **< 표 1 > 분석 과정**

[2] distance=0.9361 | chunk_id=1 | doc_type=markdown
source=data\AI@Data_Report_CLEANED.md
source_name=AI@Data_Report_CLEANED.md
AI@Data Report ** -제3호**   **목차**   ❶ **배경 및 목적 /

In [16]:
query =  "산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?"
print("유사도 거리 점수 (낮을수록 유사):")
inspect_retrieval(query, k=5)

유사도 거리 점수 (낮을수록 유사):
[질문] 산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?
[검색 개수 k] 5
ChromaDB distance score: 낮을수록 더 유사함

[1] distance=0.7977 | chunk_id=11 | doc_type=markdown
source=data\AI@Data_Report_CLEANED.md
source_name=AI@Data_Report_CLEANED.md
- - 산업·기술·정책 분야 모두에서 상위 단어가 6~8개 주제 축으로 수렴하는 패턴을 보여 7개 키워드를 채택해 포괄성과 해석성을 확보함   - - 분야 간 키워드 수를 동일하게 맞춤으로써 비교 과정에서 기준 차이에 따른 왜곡을 최소화함 본 분석의 분야별 핵심 키워드는 국내외 주요 매체에서 반복 등장한 상위 빈도 핵심어 중 산업·기술·정책의 주요 이슈와 변화 방향을 가장 정확히 대표하는 단어들을 선별하여 구성   - - 핵심 키워드는 산업·기술·정책 각 분야에서 반복적으로 등장한 상위 빈도 핵심어 중 대표성이 가장 높은 단어를 기반으로 선정함   - - 이는 각 키워드가 분야별 담론 구조와 주요 이슈·변화 방향(시장 확산, 기술 고도화, 규제 강화 등)을 가장 명확하게 설명하는 지표 역할을 하기 때문임   ## **< 표 2 > 분야별 데이터 수집 키워드**   |분야|사용 키워드| |---|---| |AI산업|AI투자,시장규모,기업도입, 생성형AI,펀딩,스타트업,산업전환| |AI기술|AI에이전트,멀티모달,LLM,온디바이스AI,오픈소스,딥시크,모델개발| |AI정책|AI규제,AI기본법,EU AI Act,거버넌스, 행정명령,데이터보호,인프라정책|  - 분야별로

[2] distance=1.0864 | chunk_id=10 | doc_type=markdown
source=data\AI@Data_Report_CLEANED.md
source_name=AI@Data_Report_CLEANED.md
- 각 분야별 담론 구조 도출을 위해 본 연구는 데

# 실행과 답변 비교

## 검색 결과와 최종 답변 확인

In [17]:
def ask_markdown_rag(question: str):
    """
    Markdown 기반 RAG 체인을 실행하고 답변을 출력한다.
    """

    answer = naive_chain.invoke(question)

    print("=" * 100)
    print(f"[질문]\n{question}")
    print("\n[Markdown 기반 RAG 답변]")
    print(answer)
    print("=" * 100)

    return answer

In [18]:
rag_questions = [
    "2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?",
    "내년 AI 모델은 어떤 방향으로 발전할 것으로 보이나요?",
    "산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?",
]

for question in rag_questions:
    ask_markdown_rag(question)

[질문]
2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?

[Markdown 기반 RAG 답변]
2026년 AI 기술 전망에서 핵심 변화는 다음과 같습니다:

1. **AI의 기업 운영 인프라로 자리 잡기**: AI 활용이 여러 산업군에서 실험 단계를 넘어 일부 핵심 공정과 서비스 운영에 본격 도입되기 시작하면서, AI가 기업 운영의 필수 요소로 자리 잡을 것으로 예상됩니다.

2. **고품질·도메인 특화 데이터의 중요성 증가**: AI 활용의 확산에 따라 고품질 데이터와 도메인 특화 데이터(버티컬 데이터)의 중요성이 크게 높아지고 있으며, 데이터 확보·관리·유통을 둘러싼 생태계가 빠르게 성장할 것으로 보입니다.

3. **기술적 고도화와 안전성 검증의 중요성**: 기술 고도화에 따라 AI의 안전성·정합성 검증의 중요성이 커지며, 신뢰성 평가와 표준화 마련이 AI 기술 확산의 기반 인프라로 자리 잡을 전망입니다.

4. **기존 산업의 가치 사슬 재편**: 기존 산업의 가치 사슬이 전면 재편되기보다는 핵심 공정 및 고부가 서비스 단에서 데이터·AI 중심 구조로 이동하는 압력이 강화될 것으로 예상됩니다.

이러한 변화들은 AI 기술의 발전과 함께 산업 전반에 걸쳐 중요한 영향을 미칠 것으로 보입니다.
[질문]
내년 AI 모델은 어떤 방향으로 발전할 것으로 보이나요?

[Markdown 기반 RAG 답변]
내년인 2026년 AI 모델은 다음과 같은 방향으로 발전할 것으로 예상됩니다:

1. **AI 활용의 확산**: AI가 여러 산업군에서 실험 단계를 넘어 일부 핵심 공정과 서비스 운영에 본격적으로 도입되기 시작할 것으로 보입니다. 이는 AI가 기업 운영 인프라로 자리 잡는 흐름을 확산시킬 것입니다.

2. **고품질 데이터의 중요성 증가**: AI 활용이 확산됨에 따라 고품질·도메인 특화 데이터(버티컬 데이터)의 중요성이 크게 높아질 것입니다. 데이터 확보, 관리, 유통을 둘러싼 생태계가 빠르게 성장할 것으로 예상됩니다.

3. **기술 고도화**: 고품

# 정리

이번 실습에서는 PDF 문서를 Markdown으로 변환한 뒤 RAG를 구성했다.  
PDF 직접 로딩 방식보다 Markdown 변환 방식은 제목, 문단, 표 구조를 더 잘 보존할 수 있다.

### 확인한 내용

1. `pymupdf4llm`을 사용하면 PDF를 Markdown 형식으로 변환할 수 있다.
2. 변환된 Markdown에는 이미지 생략 문구, 반복 제목, 깨진 줄바꿈 같은 노이즈가 포함될 수 있다.
3. Markdown 정제를 통해 불필요한 문구와 반복 헤더를 줄일 수 있다.
4. Markdown 파일은 `TextLoader`로 로딩할 수 있다.
5. `TextLoader`는 기본적으로 페이지 번호 metadata를 제공하지 않는다.
6. Markdown 기반 청크에는 `chunk_id`를 추가해 검색 결과를 추적하는 것이 좋다.
7. 표가 포함된 문서는 chunk 크기가 너무 작으면 행·열 관계가 끊길 수 있다.
8. Markdown 변환은 검색 품질을 개선할 수 있지만, 질문 표현 차이와 관련성이 낮은 청크 반환 문제는 여전히 남는다.

### 다음 실습 흐름

```text
01_naive_rag_limits                  → PDF 직접 로딩 기반 기본 RAG의 한계 확인
02_data_pdf_to_md                    → PDF를 Markdown으로 변환하여 문서 구조 개선
03_query_rewriting_ai_trend          → 질문을 검색에 적합한 형태로 재작성
04_multi_query_ai_trend              → 여러 관점의 검색 질문 생성
05_reranking_ai_trend                → 검색 결과 재순위화
06_contextual_compression_ai_trend   → 검색 문서 압축 및 필터링
07_source_answer_ai_trend            → 문서/청크 출처 기반 답변 생성
08_rag_evaluation_ai_trend           → RAG 답변 품질 평가